In [4]:

"""
NewLine Cinema Management System - Client
A Tkinter-based GUI client for cinema management operations.
"""

import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import socket
import json
import threading
from datetime import datetime
import os

class CinemaClient:
    def __init__(self, host='localhost', port=12345):
        self.host = host
        self.port = port
        self.socket = None
        
        # Initialize with your movie data
        self.movies = [
            {
                'id': 1,
                'title': 'Moana',
                'cinema_room': 1,
                'release_date': '2025-06-01',
                'end_date': '2025-06-28',
                'ticket_price': 100,
                'tickets_available': 150,
                'showtimes': ['14:00', '16:30', '19:00', '21:30']
            },
            {
                'id': 2,
                'title': 'Sing',
                'cinema_room': 2,
                'release_date': '2025-06-02',
                'end_date': '2025-06-30',
                'ticket_price': 90,
                'tickets_available': 120,
                'showtimes': ['13:30', '16:00', '18:30', '21:00']
            },
            {
                'id': 3,
                'title': 'Cinderella',
                'cinema_room': 3,
                'release_date': '2025-06-03',
                'end_date': '2025-07-01',
                'ticket_price': 85,
                'tickets_available': 130,
                'showtimes': ['14:30', '17:00', '19:30', '22:00']
            },
            {
                'id': 4,
                'title': 'The Little Mermaid',
                'cinema_room': 4,
                'release_date': '2025-06-04',
                'end_date': '2025-07-02',
                'ticket_price': 95,
                'tickets_available': 140,
                'showtimes': ['13:00', '15:30', '18:00', '20:30']
            },
            {
                'id': 5,
                'title': 'Black Panther: Wakanda Forever',
                'cinema_room': 5,
                'release_date': '2025-06-05',
                'end_date': '2025-07-03',
                'ticket_price': 110,
                'tickets_available': 160,
                'showtimes': ['15:00', '17:45', '20:30', '23:15']
            },
            {
                'id': 6,
                'title': 'Captain Marvel',
                'cinema_room': 6,
                'release_date': '2025-06-06',
                'end_date': '2025-07-04',
                'ticket_price': 100,
                'tickets_available': 135,
                'showtimes': ['14:15', '16:45', '19:15', '21:45']
            },
            {
                'id': 7,
                'title': 'Barbie',
                'cinema_room': 7,
                'release_date': '2025-06-07',
                'end_date': '2025-07-05',
                'ticket_price': 120,
                'tickets_available': 180,
                'showtimes': ['13:45', '16:15', '18:45', '21:15']
            }
        ]
        
        # Create main window
        self.root = tk.Tk()
        self.root.title("NewLine Cinema Management System")
        self.root.geometry("900x750")
        self.root.configure(bg='#2c3e50')
        
        # Configure styles
        self.setup_styles()
        
        # Create GUI components
        self.create_widgets()
        
        # Initialize the interface with movie data
        self.update_movie_combos()
        self.status_var.set(f"Loaded {len(self.movies)} movies")
    
    def setup_styles(self):
        """Configure ttk styles for better appearance."""
        style = ttk.Style()
        style.theme_use('clam')
        
        # Configure custom styles
        style.configure('Title.TLabel', font=('Arial', 16, 'bold'), foreground='white', background='#2c3e50')
        style.configure('Heading.TLabel', font=('Arial', 12, 'bold'), foreground='white', background='#2c3e50')
        style.configure('Custom.TButton', font=('Arial', 10, 'bold'))
        style.configure('Custom.TFrame', background='#34495e')
        style.configure('Custom.TLabel', foreground='white', background='#34495e')
    
    def create_widgets(self):
        """Create and arrange GUI widgets."""
        # Main title
        title_label = ttk.Label(self.root, text="NewLine Cinema Management System", 
                               style='Title.TLabel')
        title_label.pack(pady=10)
        
        # Create notebook for tabs
        notebook = ttk.Notebook(self.root)
        notebook.pack(fill='both', expand=True, padx=10, pady=5)
        
        # Ticket Purchase Tab
        self.create_purchase_tab(notebook)
        
        # Movie Management Tab
        self.create_management_tab(notebook)
        
        # Status bar
        self.status_var = tk.StringVar()
        self.status_var.set("Ready")
        status_bar = ttk.Label(self.root, textvariable=self.status_var, 
                              relief=tk.SUNKEN, anchor=tk.W)
        status_bar.pack(side=tk.BOTTOM, fill=tk.X)
    
    def create_purchase_tab(self, notebook):
        """Create the ticket purchase tab."""
        purchase_frame = ttk.Frame(notebook, style='Custom.TFrame')
        notebook.add(purchase_frame, text="Ticket Purchase")
        
        # Movie selection frame
        movie_frame = ttk.LabelFrame(purchase_frame, text="Select Movie", 
                                   style='Custom.TFrame')
        movie_frame.pack(fill='x', padx=10, pady=5)
        
        ttk.Label(movie_frame, text="Movie:", style='Custom.TLabel').grid(row=0, column=0, sticky='w', padx=5, pady=5)
        self.movie_var = tk.StringVar()
        self.movie_combo = ttk.Combobox(movie_frame, textvariable=self.movie_var, 
                                       state='readonly', width=50)
        self.movie_combo.grid(row=0, column=1, padx=5, pady=5, sticky='ew')
        self.movie_combo.bind('<<ComboboxSelected>>', self.on_movie_selected)
        
        # Showtime selection
        ttk.Label(movie_frame, text="Showtime:", style='Custom.TLabel').grid(row=1, column=0, sticky='w', padx=5, pady=5)
        self.showtime_var = tk.StringVar()
        self.showtime_combo = ttk.Combobox(movie_frame, textvariable=self.showtime_var, 
                                          state='readonly', width=20)
        self.showtime_combo.grid(row=1, column=1, padx=5, pady=5, sticky='w')
        
        # Movie details frame
        details_frame = ttk.LabelFrame(purchase_frame, text="Movie Details", 
                                     style='Custom.TFrame')
        details_frame.pack(fill='x', padx=10, pady=5)
        
        # Movie info labels
        self.movie_info_labels = {}
        info_fields = ['Room', 'Release Date', 'End Date', 'Available Tickets', 'Price']
        for i, field in enumerate(info_fields):
            ttk.Label(details_frame, text=f"{field}:", style='Custom.TLabel').grid(row=i//2, column=(i%2)*2, sticky='w', padx=5, pady=2)
            self.movie_info_labels[field] = ttk.Label(details_frame, text="", style='Custom.TLabel')
            self.movie_info_labels[field].grid(row=i//2, column=(i%2)*2+1, sticky='w', padx=5, pady=2)
        
        # Purchase frame
        purchase_detail_frame = ttk.LabelFrame(purchase_frame, text="Purchase Details", 
                                             style='Custom.TFrame')
        purchase_detail_frame.pack(fill='x', padx=10, pady=5)
        
        ttk.Label(purchase_detail_frame, text="Customer Name:", style='Custom.TLabel').grid(row=0, column=0, sticky='w', padx=5, pady=5)
        self.customer_name_var = tk.StringVar()
        self.customer_entry = ttk.Entry(purchase_detail_frame, textvariable=self.customer_name_var, width=30)
        self.customer_entry.grid(row=0, column=1, padx=5, pady=5, sticky='ew')
        
        ttk.Label(purchase_detail_frame, text="Number of Tickets:", style='Custom.TLabel').grid(row=1, column=0, sticky='w', padx=5, pady=5)
        self.tickets_var = tk.StringVar()
        self.tickets_entry = ttk.Entry(purchase_detail_frame, textvariable=self.tickets_var, width=10)
        self.tickets_entry.grid(row=1, column=1, padx=5, pady=5, sticky='w')
        self.tickets_entry.bind('<KeyRelease>', self.calculate_total)
        
        ttk.Label(purchase_detail_frame, text="Total Cost:", style='Custom.TLabel').grid(row=2, column=0, sticky='w', padx=5, pady=5)
        self.total_var = tk.StringVar()
        self.total_label = ttk.Label(purchase_detail_frame, textvariable=self.total_var, 
                                   style='Custom.TLabel', font=('Arial', 12, 'bold'))
        self.total_label.grid(row=2, column=1, padx=5, pady=5, sticky='w')
        
        # Buttons frame
        button_frame = ttk.Frame(purchase_frame, style='Custom.TFrame')
        button_frame.pack(fill='x', padx=10, pady=10)
        
        ttk.Button(button_frame, text="Purchase Tickets", command=self.purchase_tickets,
                  style='Custom.TButton').pack(side='left', padx=5)
        ttk.Button(button_frame, text="Refresh Movies", command=self.refresh_movies,
                  style='Custom.TButton').pack(side='left', padx=5)
    
    def create_management_tab(self, notebook):
        """Create the movie management tab."""
        mgmt_frame = ttk.Frame(notebook, style='Custom.TFrame')
        notebook.add(mgmt_frame, text="Movie Management")
        
        # Add movie frame
        add_frame = ttk.LabelFrame(mgmt_frame, text="Add New Movie", style='Custom.TFrame')
        add_frame.pack(fill='x', padx=10, pady=5)
        
        # Entry fields for new movie
        self.new_movie_vars = {}
        fields = [
            ('Title', 'title'),
            ('Cinema Room (1-7)', 'cinema_room'),
            ('Release Date (YYYY-MM-DD)', 'release_date'),
            ('End Date (YYYY-MM-DD)', 'end_date'),
            ('Available Tickets', 'tickets_available'),
            ('Ticket Price (R)', 'ticket_price')
        ]
        
        for i, (label, var_name) in enumerate(fields):
            ttk.Label(add_frame, text=f"{label}:", style='Custom.TLabel').grid(row=i//2, column=(i%2)*2, sticky='w', padx=5, pady=2)
            self.new_movie_vars[var_name] = tk.StringVar()
            entry = ttk.Entry(add_frame, textvariable=self.new_movie_vars[var_name], width=25)
            entry.grid(row=i//2, column=(i%2)*2+1, padx=5, pady=2, sticky='ew')
        
        ttk.Button(add_frame, text="Add Movie", command=self.add_movie,
                  style='Custom.TButton').grid(row=3, column=0, columnspan=4, pady=10)
        
        # Update/Delete movie frame
        update_frame = ttk.LabelFrame(mgmt_frame, text="Update/Delete Movie", style='Custom.TFrame')
        update_frame.pack(fill='x', padx=10, pady=5)
        
        ttk.Label(update_frame, text="Select Movie:", style='Custom.TLabel').grid(row=0, column=0, sticky='w', padx=5, pady=5)
        self.update_movie_var = tk.StringVar()
        self.update_movie_combo = ttk.Combobox(update_frame, textvariable=self.update_movie_var,
                                             state='readonly', width=50)
        self.update_movie_combo.grid(row=0, column=1, padx=5, pady=5, sticky='ew')
        self.update_movie_combo.bind('<<ComboboxSelected>>', self.on_update_movie_selected)
        
        # Update fields
        self.update_movie_vars = {}
        for i, (label, var_name) in enumerate(fields):
            ttk.Label(update_frame, text=f"{label}:", style='Custom.TLabel').grid(row=i//2+1, column=(i%2)*2, sticky='w', padx=5, pady=2)
            self.update_movie_vars[var_name] = tk.StringVar()
            entry = ttk.Entry(update_frame, textvariable=self.update_movie_vars[var_name], width=25)
            entry.grid(row=i//2+1, column=(i%2)*2+1, padx=5, pady=2, sticky='ew')
        
        # Update/Delete buttons
        button_frame = ttk.Frame(update_frame, style='Custom.TFrame')
        button_frame.grid(row=4, column=0, columnspan=4, pady=10)
        
        ttk.Button(button_frame, text="Update Movie", command=self.update_movie,
                  style='Custom.TButton').pack(side='left', padx=5)
        ttk.Button(button_frame, text="Delete Movie", command=self.delete_movie,
                  style='Custom.TButton').pack(side='left', padx=5)
        ttk.Button(button_frame, text="Clear Fields", command=self.clear_update_fields,
                  style='Custom.TButton').pack(side='left', padx=5)
    
    def connect_to_server(self):
        """Establish connection to the server."""
        try:
            self.socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.socket.connect((self.host, self.port))
            self.status_var.set("Connected to server")
        except Exception as e:
            self.status_var.set("Using local data (server not available)")
    
    def send_request(self, request):
        """Send request to server and return response."""
        # Since we're using local data, simulate server responses
        if request['action'] == 'get_movies':
            return {'status': 'success', 'data': self.movies}
        else:
            return {'status': 'error', 'message': 'Server functionality not available in local mode'}
    
    def refresh_movies(self):
        """Refresh the movie list."""
        self.update_movie_combos()
        self.status_var.set(f"Refreshed - {len(self.movies)} movies available")
    
    def update_movie_combos(self):
        """Update the movie combo boxes with current movie list."""
        movie_titles = [f"{movie['title']} (Room {movie['cinema_room']})" for movie in self.movies]
        
        self.movie_combo['values'] = movie_titles
        self.update_movie_combo['values'] = movie_titles
        
        if movie_titles:
            self.movie_combo.set(movie_titles[0])
            self.on_movie_selected(None)
    
    def on_movie_selected(self, event):
        """Handle movie selection in purchase tab."""
        selection = self.movie_var.get()
        if not selection:
            return
        
        # Find selected movie
        selected_movie = None
        for movie in self.movies:
            if f"{movie['title']} (Room {movie['cinema_room']})" == selection:
                selected_movie = movie
                break
        
        if selected_movie:
            # Update movie info
            self.movie_info_labels['Room'].config(text=str(selected_movie['cinema_room']))
            self.movie_info_labels['Release Date'].config(text=selected_movie['release_date'])
            self.movie_info_labels['End Date'].config(text=selected_movie['end_date'])
            self.movie_info_labels['Available Tickets'].config(text=str(selected_movie['tickets_available']))
            self.movie_info_labels['Price'].config(text=f"R{selected_movie['ticket_price']:.2f}")
            
            # Update showtime combo
            self.showtime_combo['values'] = selected_movie['showtimes']
            if selected_movie['showtimes']:
                self.showtime_combo.set(selected_movie['showtimes'][0])
            
            # Calculate total if tickets are entered
            self.calculate_total(None)
    
    def on_update_movie_selected(self, event):
        """Handle movie selection in management tab."""
        selection = self.update_movie_var.get()
        if not selection:
            return
        
        # Find selected movie
        selected_movie = None
        for movie in self.movies:
            if f"{movie['title']} (Room {movie['cinema_room']})" == selection:
                selected_movie = movie
                break
        
        if selected_movie:
            self.update_movie_vars['title'].set(selected_movie['title'])
            self.update_movie_vars['cinema_room'].set(str(selected_movie['cinema_room']))
            self.update_movie_vars['release_date'].set(selected_movie['release_date'])
            self.update_movie_vars['end_date'].set(selected_movie['end_date'])
            self.update_movie_vars['tickets_available'].set(str(selected_movie['tickets_available']))
            self.update_movie_vars['ticket_price'].set(str(selected_movie['ticket_price']))
    
    def calculate_total(self, event):
        """Calculate total cost based on number of tickets."""
        try:
            tickets = int(self.tickets_var.get() or 0)
            selection = self.movie_var.get()
            
            if tickets > 0 and selection:
                # Find selected movie
                for movie in self.movies:
                    if f"{movie['title']} (Room {movie['cinema_room']})" == selection:
                        total = tickets * movie['ticket_price']
                        self.total_var.set(f"R{total:.2f}")
                        return
            
            self.total_var.set("R0.00")
        except ValueError:
            self.total_var.set("R0.00")
    
    def purchase_tickets(self):
        """Handle ticket purchase."""
        # Validate inputs
        customer_name = self.customer_name_var.get().strip()
        if not customer_name:
            messagebox.showerror("Error", "Please enter customer name")
            return
        
        try:
            num_tickets = int(self.tickets_var.get())
            if num_tickets <= 0:
                raise ValueError()
        except ValueError:
            messagebox.showerror("Error", "Please enter a valid number of tickets")
            return
        
        selection = self.movie_var.get()
        showtime = self.showtime_var.get()
        if not selection:
            messagebox.showerror("Error", "Please select a movie")
            return
        
        if not showtime:
            messagebox.showerror("Error", "Please select a showtime")
            return
        
        # Find selected movie
        selected_movie = None
        for movie in self.movies:
            if f"{movie['title']} (Room {movie['cinema_room']})" == selection:
                selected_movie = movie
                break
        
        if not selected_movie:
            messagebox.showerror("Error", "Selected movie not found")
            return
        
        if num_tickets > selected_movie['tickets_available']:
            messagebox.showerror("Error", f"Only {selected_movie['tickets_available']} tickets available")
            return
        
        # Calculate total
        total = num_tickets * selected_movie['ticket_price']
        
        # Update available tickets
        selected_movie['tickets_available'] -= num_tickets
        
        # Generate receipt
        self.generate_receipt(selected_movie, customer_name, num_tickets, total, showtime)
        
        # Clear form
        self.customer_name_var.set("")
        self.tickets_var.set("")
        self.total_var.set("R0.00")
        
        # Refresh movies to update available tickets
        self.refresh_movies()
        
        messagebox.showinfo("Success", 
                          f"Purchase successful!\nTotal: R{total:.2f}\n"
                          f"Showtime: {showtime}\n"
                          f"Remaining tickets: {selected_movie['tickets_available']}")
        
        self.status_var.set("Purchase completed")
    
    def generate_receipt(self, movie, customer_name, num_tickets, total, showtime):
        """Generate and save a receipt file."""
        try:
            receipt_content = f"""
============================================
         NEWLINE CINEMA RECEIPT
============================================

Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Movie ID: {movie['id']}
Movie Title: {movie['title']}
Cinema Room: {movie['cinema_room']}
Showtime: {showtime}

Customer Name: {customer_name}
Number of Tickets: {num_tickets}
Price per Ticket: R{movie['ticket_price']:.2f}
Total Amount: R{total:.2f}

============================================
        Thank you for your purchase!
============================================
"""
            
            # Create receipts directory if it doesn't exist
            if not os.path.exists('receipts'):
                os.makedirs('receipts')
            
            # Generate filename
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"receipts/receipt_{timestamp}_{customer_name.replace(' ', '_')}.txt"
            
            # Save receipt
            with open(filename, 'w') as f:
                f.write(receipt_content)
            
            # Ask user if they want to open the receipt
            if messagebox.askyesno("Receipt Saved", f"Receipt saved as {filename}\nWould you like to open it?"):
                if os.name == 'nt':  # Windows
                    os.startfile(filename)
                else:  # Mac/Linux
                    os.system(f'open {filename}')
                
        except Exception as e:
            messagebox.showwarning("Receipt Error", f"Failed to save receipt: {e}")
    
    def add_movie(self):
        """Add a new movie to the database."""
        # Validate inputs
        movie_data = {}
        try:
            movie_data['title'] = self.new_movie_vars['title'].get().strip()
            if not movie_data['title']:
                raise ValueError("Title is required")
            
            movie_data['cinema_room'] = int(self.new_movie_vars['cinema_room'].get())
            if not (1 <= movie_data['cinema_room'] <= 7):
                raise ValueError("Cinema room must be between 1 and 7")
            
            movie_data['release_date'] = self.new_movie_vars['release_date'].get().strip()
            movie_data['end_date'] = self.new_movie_vars['end_date'].get().strip()
            
            if not movie_data['release_date'] or not movie_data['end_date']:
                raise ValueError("Release date and end date are required")
            
            movie_data['tickets_available'] = int(self.new_movie_vars['tickets_available'].get())
            if movie_data['tickets_available'] < 0:
                raise ValueError("Tickets available cannot be negative")
            
            movie_data['ticket_price'] = float(self.new_movie_vars['ticket_price'].get())
            if movie_data['ticket_price'] <= 0:
                raise ValueError("Ticket price must be positive")
            
            # Generate new ID
            movie_data['id'] = max([m['id'] for m in self.movies]) + 1 if self.movies else 1
            movie_data['showtimes'] = ['14:00', '16:30', '19:00', '21:30']  # Default showtimes
                
        except ValueError as e:
            messagebox.showerror("Input Error", str(e))
            return
        
        # Add movie to list
        self.movies.append(movie_data)
        
        # Clear form
        for var in self.new_movie_vars.values():
            var.set("")
        
        # Refresh movies
        self.refresh_movies()
        
        messagebox.showinfo("Success", "Movie added successfully!")
        self.status_var.set("Movie added")
    
    def update_movie(self):
        """Update an existing movie."""
        selection = self.update_movie_var.get()
        if not selection:
            messagebox.showerror("Error", "Please select a movie to update")
            return
        
        # Find selected movie
        selected_movie = None
        for movie in self.movies:
            if f"{movie['title']} (Room {movie['cinema_room']})" == selection:
                selected_movie = movie
                break
        
        if not selected_movie:
            messagebox.showerror("Error", "Selected movie not found")
            return
        
        try:
            # Update fields that have values
            if self.update_movie_vars['title'].get().strip():
                selected_movie['title'] = self.update_movie_vars['title'].get().strip()
            
            if self.update_movie_vars['cinema_room'].get().strip():
                room = int(self.update_movie_vars['cinema_room'].get())
                if not (1 <= room <= 7):
                    raise ValueError("Cinema room must be between 1 and 7")
                selected_movie['cinema_room'] = room
            
            if self.update_movie_vars['release_date'].get().strip():
                selected_movie['release_date'] = self.update_movie_vars['release_date'].get().strip()
            
            if self.update_movie_vars['end_date'].get().strip():
                selected_movie['end_date'] = self.update_movie_vars['end_date'].get().strip()
            
            if self.update_movie_vars['tickets_available'].get().strip():
                tickets = int(self.update_movie_vars['tickets_available'].get())
                if tickets < 0:
                    raise ValueError("Tickets available cannot be negative")
                selected_movie['tickets_available'] = tickets
            
            if self.update_movie_vars['ticket_price'].get().strip():
                price = float(self.update_movie_vars['ticket_price'].get())
                if price <= 0:
                    raise ValueError("Ticket price must be positive")
                selected_movie['ticket_price'] = price
                
        except ValueError as e:
            messagebox.showerror("Input Error", str(e))
            return
        
        # Clear form
        self.clear_update_fields()
        
        # Refresh movies
        self.refresh_movies()
        
        messagebox.showinfo("Success", "Movie updated successfully!")
        self.status_var.set("Movie updated")
    
    def delete_movie(self):
        """Delete a movie from the database."""
        selection = self.update_movie_var.get()
        if not selection:
            messagebox.showerror("Error", "Please select a movie to delete")
            return
        
        # Find selected movie
        selected_movie = None
        for i, movie in enumerate(self.movies):
            if f"{movie['title']} (Room {movie['cinema_room']})" == selection:
                selected_movie = (i, movie)
                break
        
        if not selected_movie:
            messagebox.showerror("Error", "Selected movie not found")
            return
        
        # Confirm deletion
        if not messagebox.askyesno("Confirm Delete", 
                                  f"Are you sure you want to delete '{selection}'?\n"
                                  "This action cannot be undone."):
            return
        
        # Remove movie from list
        self.movies.pop(selected_movie[0])
        
        # Clear form
        self.clear_update_fields()
        
        # Refresh movies
        self.refresh_movies()
        
        messagebox.showinfo("Success", "Movie deleted successfully!")
        self.status_var.set("Movie deleted")
    
    def clear_update_fields(self):
        """Clear all update form fields."""
        for var in self.update_movie_vars.values():
            var.set("")
        self.update_movie_var.set("")
    
    def on_closing(self):
        """Handle application closing."""
        if self.socket:
            try:
                self.socket.close()
            except:
                pass
        self.root.destroy()
    
    def run(self):
        """Start the GUI application."""
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        self.root.mainloop()

if __name__ == "__main__":
    try:
        client = CinemaClient()
        client.run()
    except Exception as e:
        print(f"Application error: {e}")
        # Create a temporary root for the error message
        root = tk.Tk()
        root.withdraw()  # Hide the empty window
        messagebox.showerror("Application Error", f"Failed to start application: {e}")
        root.destroy()